In [0]:
USE CATALOG tailoring_lakehouse;

In [0]:
CREATE OR REPLACE TABLE gold.fact_order_status
USING DELTA
AS

WITH status_summary AS (

    SELECT
        order_id,

        MIN(changed_on) AS order_started_at,

        MAX(
            CASE
                WHEN status = 'COMPLETED'
                THEN changed_on
            END
        ) AS completed_at,

        COUNT(*) AS status_change_count

    FROM silver.order_status_history

    GROUP BY order_id
),

latest_status AS (

    SELECT
        order_id,
        status AS latest_status,
        changed_on AS latest_status_at,

        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY changed_on DESC
        ) AS rn

    FROM silver.order_status_history
)

SELECT

    f.order_id,

    f.customer_id,

    f.measurement_id,

    f.garment_id,

    f.tailor_id,

    f.deadline_date,

    f.order_amount,

    f.order_status,

    s.order_started_at,

    s.completed_at,

    l.latest_status,

    l.latest_status_at,

    s.status_change_count,

    CASE
        WHEN s.completed_at IS NOT NULL
        THEN ROUND(
            (
                UNIX_TIMESTAMP(s.completed_at)
                - UNIX_TIMESTAMP(s.order_started_at)
            ) / 86400.0,
            2
        )
    END AS turnaround_days,

    CASE
        WHEN s.completed_at IS NOT NULL
        THEN DATEDIFF(
            DATE(s.completed_at),
            f.deadline_date
        )
    END AS days_late,

    CASE
        WHEN s.completed_at IS NULL
             AND CURRENT_DATE() > f.deadline_date
            THEN 'OVERDUE_PENDING'

        WHEN s.completed_at IS NULL
            THEN 'PENDING'

        WHEN DATE(s.completed_at) > f.deadline_date
            THEN 'OVERDUE_COMPLETED'

        ELSE 'COMPLETED_ON_TIME'
    END AS delivery_status

FROM gold.fact_orders f

LEFT JOIN status_summary s
    ON f.order_id = s.order_id

LEFT JOIN latest_status l
    ON f.order_id = l.order_id
    AND l.rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM gold.fact_order_status
ORDER BY order_id;

order_id,customer_id,measurement_id,garment_id,tailor_id,deadline_date,order_amount,order_status,order_started_at,completed_at,latest_status,latest_status_at,status_change_count,turnaround_days,days_late,delivery_status
1,1,1,1,1001,2024-04-12,250.00,COMPLETED,2024-04-05T09:15:00.000Z,2024-04-11T17:30:00.000Z,COMPLETED,2024-04-11T17:30:00.000Z,2,6.34,-1,COMPLETED_ON_TIME
2,2,2,2,1002,2024-05-15,2500.00,COMPLETED,2024-05-05T10:00:00.000Z,2024-05-14T18:00:00.000Z,COMPLETED,2024-05-14T18:00:00.000Z,3,9.33,-1,COMPLETED_ON_TIME
3,3,3,3,1003,2024-05-25,500.00,COMPLETED,2024-05-15T11:00:00.000Z,2024-05-24T16:45:00.000Z,COMPLETED,2024-05-24T16:45:00.000Z,2,9.24,-1,COMPLETED_ON_TIME
4,4,4,4,1004,2024-04-13,1000.00,PENDING,2024-04-05T12:30:00.000Z,null,PENDING,2024-04-05T12:30:00.000Z,1,null,null,OVERDUE_PENDING
5,5,5,5,1005,2024-07-13,100.00,PENDING,2024-07-01T09:30:00.000Z,null,PENDING,2024-07-01T09:30:00.000Z,1,null,null,OVERDUE_PENDING
6,6,6,6,1006,2024-06-23,1500.00,COMPLETED,2024-06-15T10:00:00.000Z,2024-06-22T17:00:00.000Z,COMPLETED,2024-06-22T17:00:00.000Z,3,7.29,-1,COMPLETED_ON_TIME
7,7,7,7,1007,2024-06-20,700.00,PENDING,2024-06-10T11:15:00.000Z,null,PENDING,2024-06-10T11:15:00.000Z,1,null,null,OVERDUE_PENDING
8,8,8,8,1008,2024-05-23,1700.00,PENDING,2024-05-15T09:45:00.000Z,null,IN_PROGRESS,2024-05-19T15:30:00.000Z,2,null,null,OVERDUE_PENDING
9,9,9,9,1009,2024-04-14,400.00,COMPLETED,2024-04-08T10:20:00.000Z,2024-04-13T17:15:00.000Z,COMPLETED,2024-04-13T17:15:00.000Z,2,5.29,-1,COMPLETED_ON_TIME
10,10,10,10,1010,2024-04-14,400.00,COMPLETED,2024-04-08T11:00:00.000Z,2024-04-13T16:30:00.000Z,COMPLETED,2024-04-13T16:30:00.000Z,2,5.23,-1,COMPLETED_ON_TIME


In [0]:
SELECT
    order_id,
    order_started_at,
    completed_at,
    turnaround_days
FROM gold.fact_order_status
WHERE completed_at IS NOT NULL
ORDER BY turnaround_days DESC;

order_id,order_started_at,completed_at,turnaround_days
2,2024-05-05T10:00:00.000Z,2024-05-14T18:00:00.000Z,9.33
20,2024-10-08T10:00:00.000Z,2024-10-17T17:30:00.000Z,9.31
3,2024-05-15T11:00:00.000Z,2024-05-24T16:45:00.000Z,9.24
11,2024-08-05T09:00:00.000Z,2024-08-12T18:15:00.000Z,7.39
16,2024-09-10T09:45:00.000Z,2024-09-17T17:45:00.000Z,7.33
18,2024-09-25T11:00:00.000Z,2024-10-02T18:00:00.000Z,7.29
6,2024-06-15T10:00:00.000Z,2024-06-22T17:00:00.000Z,7.29
12,2024-08-12T10:30:00.000Z,2024-08-19T17:00:00.000Z,7.27
1,2024-04-05T09:15:00.000Z,2024-04-11T17:30:00.000Z,6.34
15,2024-09-05T10:00:00.000Z,2024-09-11T18:00:00.000Z,6.33


In [0]:
SELECT
    order_id,
    deadline_date,
    completed_at,
    days_late,
    delivery_status
FROM gold.fact_order_status
ORDER BY days_late DESC;

order_id,deadline_date,completed_at,days_late,delivery_status
16,2024-09-18,2024-09-17T17:45:00.000Z,-1,COMPLETED_ON_TIME
6,2024-06-23,2024-06-22T17:00:00.000Z,-1,COMPLETED_ON_TIME
9,2024-04-14,2024-04-13T17:15:00.000Z,-1,COMPLETED_ON_TIME
20,2024-10-18,2024-10-17T17:30:00.000Z,-1,COMPLETED_ON_TIME
1,2024-04-12,2024-04-11T17:30:00.000Z,-1,COMPLETED_ON_TIME
15,2024-09-12,2024-09-11T18:00:00.000Z,-1,COMPLETED_ON_TIME
3,2024-05-25,2024-05-24T16:45:00.000Z,-1,COMPLETED_ON_TIME
18,2024-10-03,2024-10-02T18:00:00.000Z,-1,COMPLETED_ON_TIME
11,2024-08-13,2024-08-12T18:15:00.000Z,-1,COMPLETED_ON_TIME
10,2024-04-14,2024-04-13T16:30:00.000Z,-1,COMPLETED_ON_TIME


In [0]:
SELECT

    COUNT(*) AS total_orders,

    SUM(
        CASE
            WHEN latest_status = 'COMPLETED'
            THEN 1 ELSE 0
        END
    ) AS completed_orders,

    SUM(
        CASE
            WHEN latest_status = 'PENDING'
            THEN 1 ELSE 0
        END
    ) AS pending_orders,

    SUM(
        CASE
            WHEN delivery_status IN (
                'OVERDUE_PENDING',
                'OVERDUE_COMPLETED'
            )
            THEN 1 ELSE 0
        END
    ) AS overdue_orders,

    ROUND(
        AVG(turnaround_days),
        2
    ) AS avg_turnaround_days,

    ROUND(
        AVG(
            CASE
                WHEN days_late > 0
                THEN days_late
            END
        ),
        2
    ) AS avg_days_late

FROM gold.fact_order_status;

total_orders,completed_orders,pending_orders,overdue_orders,avg_turnaround_days,avg_days_late
20,12,4,8,7.30,null


In [0]:
SELECT
    t.tailor_name,

    COUNT(f.order_id) AS total_orders,

    SUM(
        CASE
            WHEN f.latest_status = 'COMPLETED'
            THEN 1 ELSE 0
        END
    ) AS completed_orders,

    SUM(
        CASE
            WHEN f.latest_status = 'PENDING'
            THEN 1 ELSE 0
        END
    ) AS pending_orders,

    ROUND(
        AVG(f.turnaround_days),
        2
    ) AS avg_turnaround_days,

    SUM(
        CASE
            WHEN f.delivery_status IN (
                'OVERDUE_PENDING',
                'OVERDUE_COMPLETED'
            )
            THEN 1 ELSE 0
        END
    ) AS overdue_orders,

    ROUND(
        SUM(f.order_amount),
        2
    ) AS total_revenue

FROM gold.fact_order_status f

JOIN gold.dim_tailor t
    ON f.tailor_id = t.tailor_id

GROUP BY
    t.tailor_name

ORDER BY
    total_revenue DESC;

tailor_name,total_orders,completed_orders,pending_orders,avg_turnaround_days,overdue_orders,total_revenue
Rani,1,1,0,6.33,0,7800.00
Deepa,1,0,1,null,1,7000.00
Varsha,1,1,0,9.31,0,6500.00
Pooja,1,1,0,7.39,0,4800.00
Sneha,1,0,0,null,1,4200.00
Aarti,1,0,0,null,1,3500.00
Kesha,1,1,0,9.33,0,2500.00
Manju,1,1,0,7.33,0,2200.00
Lakshmi,1,0,0,null,1,1700.00
Meera,1,1,0,7.29,0,1500.00


In [0]:
CREATE OR REPLACE VIEW gold.vw_business_kpis AS

SELECT

    COUNT(*) AS total_orders,

    ROUND(
        SUM(order_amount),
        2
    ) AS total_revenue,

    ROUND(
        AVG(order_amount),
        2
    ) AS average_order_value,

    SUM(
        CASE
            WHEN latest_status = 'COMPLETED'
            THEN 1 ELSE 0
        END
    ) AS completed_orders,

    SUM(
        CASE
            WHEN latest_status = 'PENDING'
            THEN 1 ELSE 0
        END
    ) AS pending_orders,

    SUM(
        CASE
            WHEN delivery_status IN (
                'OVERDUE_PENDING',
                'OVERDUE_COMPLETED'
            )
            THEN 1 ELSE 0
        END
    ) AS overdue_orders,

    ROUND(
        AVG(turnaround_days),
        2
    ) AS average_turnaround_days

FROM gold.fact_order_status;

In [0]:
SELECT *
FROM gold.vw_business_kpis;

total_orders,total_revenue,average_order_value,completed_orders,pending_orders,overdue_orders,average_turnaround_days
20,48070.00,2403.50,12,4,8,7.30
